# Notebook 1: RAG Fundamentals & Basic Implementation

**Duration:** 90-120 minutes  
**Level:** Intermediate  

---

## Learning Objectives

By the end of this notebook, you will be able to:

- ✅ Understand core RAG architecture and components
- ✅ Implement a basic RAG pipeline using Haystack
- ✅ Work with document stores, retrievers, and generators
- ✅ Master prompt engineering for RAG applications
- ✅ Understand when to use RAG vs fine-tuning

---

## Table of Contents

1. [Introduction to RAG](#1-introduction-to-rag)
2. [Environment Setup](#2-environment-setup)
3. [Understanding RAG Components](#3-understanding-rag-components)
4. [Data Preparation](#4-data-preparation)
5. [Example 1: Simple Q&A System](#5-example-1-simple-qa-system)
6. [Example 2: Technical Documentation Assistant](#6-example-2-technical-documentation-assistant)
7. [Example 3: News Article Summarization with Citations](#7-example-3-news-article-summarization-with-citations)
8. [Prompt Engineering Best Practices](#8-prompt-engineering-best-practices)
9. [Basic Evaluation](#9-basic-evaluation)
10. [Exercises & Challenges](#10-exercises-challenges)
11. [Summary & Next Steps](#11-summary-next-steps)

---

## 1. Introduction to RAG

### What is RAG?

**Retrieval-Augmented Generation (RAG)** is an AI architecture that combines:
- **Large Language Models (LLMs)** for natural language understanding and generation
- **External knowledge retrieval** from document stores or databases

### Why RAG?

| Problem | RAG Solution |
|---------|-------------|
| LLMs have knowledge cutoff dates | Access to up-to-date information |
| LLMs can hallucinate facts | Grounded responses with source citations |
| Domain-specific knowledge needed | Connect to proprietary knowledge bases |
| Expensive to fine-tune for every update | Simply update document store |

### RAG vs Fine-Tuning

**Use RAG when:**
- Information changes frequently
- Need source citations/transparency
- Have document-based knowledge
- Want cost-effective updates

**Use Fine-Tuning when:**
- Need specific tone/style adaptation
- Require specialized reasoning patterns
- Have limited inference latency budget
- Knowledge is stable

### Basic RAG Architecture

```
User Query → Retriever → Relevant Docs → Prompt + Docs → LLM → Answer
                ↑
           Document Store
```

### Components in Haystack

1. **Document Store**: Storage for your documents
2. **Retriever**: Finds relevant documents for a query
3. **Prompt Builder**: Constructs prompts with retrieved context
4. **Generator**: LLM that generates the final answer
5. **Pipeline**: Orchestrates the flow between components

## 2. Environment Setup

First, let's import all necessary libraries and configure our environment.

In [2]:
# Standard library imports
import os
import json
import time
from typing import List, Dict, Any, Optional
from datetime import datetime

# Third-party imports
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from tqdm.auto import tqdm

# Haystack imports
from haystack import Document, Pipeline
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack.components.generators import OpenAIGenerator
from haystack.components.builders import PromptBuilder
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.dataclasses import ChatMessage
from haystack.utils import Secret

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("✅ All imports successful!")

✅ All imports successful!


In [3]:
# Load environment variables
load_dotenv()

# Verify API key is set
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError(
        "⚠️ OPENAI_API_KEY not found!\n"
        "Please create a .env file with your OpenAI API key.\n"
        "Example: OPENAI_API_KEY=sk-..."
    )

print("✅ Environment configured successfully!")
print(f"📍 API Key found: {OPENAI_API_KEY[:20]}...")

✅ Environment configured successfully!
📍 API Key found: sk-proj-cHqt6nRJj9Nu...


## 3. Understanding RAG Components

Let's explore each component in detail with simple examples.

### 3.1 Document Representation

In Haystack, documents are represented using the `Document` dataclass.

In [4]:
# Creating a simple document
doc = Document(
    content="Haystack is an open-source framework for building LLM applications.",
    meta={
        "source": "documentation",
        "category": "introduction",
        "date": "2025-01-01"
    }
)

print("Document ID:", doc.id)
print("Content:", doc.content)
print("Metadata:", doc.meta)

Document ID: 765db240323e56967d8da988e047efcdc3610cf8615e8f31dfb3eae69e1dea2a
Content: Haystack is an open-source framework for building LLM applications.
Metadata: {'source': 'documentation', 'category': 'introduction', 'date': '2025-01-01'}


### 3.2 Document Store

The Document Store is where all documents are stored. We'll use `InMemoryDocumentStore` for simplicity.

In [5]:
# Initialize an in-memory document store
document_store = InMemoryDocumentStore()

# Create sample documents
sample_docs = [
    Document(content="Python is a high-level programming language."),
    Document(content="Machine learning is a subset of artificial intelligence."),
    Document(content="RAG combines retrieval with generation for better LLM responses.")
]

# Write documents to store
document_store.write_documents(sample_docs)

print(f"✅ Stored {document_store.count_documents()} documents")
print("\nDocuments in store:")
for doc in document_store.filter_documents():
    print(f"  - {doc.content[:50]}...")

✅ Stored 3 documents

Documents in store:
  - Python is a high-level programming language....
  - Machine learning is a subset of artificial intelli...
  - RAG combines retrieval with generation for better ...


### 3.3 Retriever

The Retriever finds relevant documents based on a query. We'll use BM25 (keyword-based) retrieval.

In [6]:
# Initialize BM25 retriever
retriever = InMemoryBM25Retriever(document_store=document_store)

# Test retrieval
query = "What is RAG?"
result = retriever.run(query=query, top_k=2)

print(f"Query: '{query}'\n")
print(f"Retrieved {len(result['documents'])} documents:\n")
for i, doc in enumerate(result['documents'], 1):
    print(f"{i}. {doc.content}")
    print(f"   Score: {doc.score:.4f}\n")

Query: 'What is RAG?'

Retrieved 2 documents:

1. RAG combines retrieval with generation for better LLM responses.
   Score: 1.3718

2. Python is a high-level programming language.
   Score: 1.1878



### 3.4 Prompt Builder

The Prompt Builder constructs prompts by combining the query with retrieved documents.

In [7]:
# Define a prompt template
template = """
Answer the question based on the context provided.

Context:
{% for doc in documents %}
  {{ doc.content }}
{% endfor %}

Question: {{ query }}

Answer:
"""

prompt_builder = PromptBuilder(template=template)

# Build a prompt
prompt_result = prompt_builder.run(
    documents=result['documents'],
    query=query
)

print("Generated Prompt:")
print("=" * 50)
print(prompt_result['prompt'])
print("=" * 50)

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


Generated Prompt:

Answer the question based on the context provided.

Context:

  RAG combines retrieval with generation for better LLM responses.

  Python is a high-level programming language.


Question: What is RAG?

Answer:


### 3.5 Generator

The Generator (LLM) produces the final answer based on the prompt.

In [8]:
# Initialize OpenAI generator
generator = OpenAIGenerator(
    api_key=Secret.from_env_var("OPENAI_API_KEY"),
    model="gpt-3.5-turbo",
    generation_kwargs={"temperature": 0.3}
)

# Generate answer
answer = generator.run(prompt=prompt_result['prompt'])

print("Generated Answer:")
print("=" * 50)
print(answer['replies'][0])
print("=" * 50)

Generated Answer:
RAG is a method that combines retrieval with generation for better responses in Language Model (LLM).


## 4. Data Preparation

Let's create helper functions to generate sample data for our examples.

In [9]:
def generate_company_knowledge_base() -> List[Document]:
    """
    Generate a synthetic company knowledge base with policies and information.
    
    Returns:
        List of Document objects representing company policies
    """
    
    company_data = [
        {
            "title": "Remote Work Policy",
            "content": """TechCorp allows employees to work remotely up to 3 days per week. 
            Remote work requests must be approved by direct managers. Employees must be available 
            during core hours (10 AM - 3 PM local time) and maintain productivity standards. 
            Equipment for remote work is provided by the company.""",
            "category": "HR",
            "last_updated": "2025-01-15"
        },
        {
            "title": "Vacation Policy",
            "content": """Full-time employees receive 20 days of paid vacation annually. 
            Vacation days accrue monthly (1.67 days per month). Employees must submit vacation 
            requests at least 2 weeks in advance through the HR portal. Unused vacation days 
            can be carried over up to 5 days into the next year.""",
            "category": "HR",
            "last_updated": "2025-01-10"
        },
        {
            "title": "Health Insurance",
            "content": """TechCorp offers comprehensive health insurance including medical, 
            dental, and vision coverage. The company covers 80% of premium costs for employees 
            and 50% for dependents. Insurance becomes effective on the first day of the month 
            following 30 days of employment.""",
            "category": "Benefits",
            "last_updated": "2025-01-20"
        },
        {
            "title": "Professional Development",
            "content": """Employees have access to $2,000 annually for professional development, 
            including conferences, courses, and certifications. Requests must be submitted through 
            the learning management system and approved by managers. Employees must remain with 
            the company for 12 months after using these funds or repay the amount.""",
            "category": "Learning",
            "last_updated": "2025-01-05"
        },
        {
            "title": "Code of Conduct",
            "content": """All employees must adhere to TechCorp's code of conduct, which includes 
            treating colleagues with respect, maintaining confidentiality, avoiding conflicts of 
            interest, and reporting unethical behavior. Violations may result in disciplinary 
            action up to and including termination.""",
            "category": "Compliance",
            "last_updated": "2024-12-01"
        },
        {
            "title": "Equipment and Technology",
            "content": """TechCorp provides all necessary equipment including laptops, monitors, 
            and software licenses. Employees can choose between Mac or Windows laptops. 
            Equipment is refreshed every 3 years. Personal use of company equipment is permitted 
            within reasonable limits.""",
            "category": "IT",
            "last_updated": "2025-01-12"
        },
        {
            "title": "Performance Reviews",
            "content": """Performance reviews are conducted bi-annually in January and July. 
            Reviews include self-assessment, peer feedback, and manager evaluation. Performance 
            ratings directly impact compensation adjustments and promotion eligibility. 
            Employees receive written feedback and development plans.""",
            "category": "HR",
            "last_updated": "2025-01-08"
        },
        {
            "title": "Security Protocols",
            "content": """All employees must use two-factor authentication for company systems. 
            Passwords must be at least 12 characters with complexity requirements. Employees 
            must not share credentials or leave workstations unlocked. Security incidents 
            must be reported immediately to the IT security team.""",
            "category": "Security",
            "last_updated": "2025-01-18"
        }
    ]
    
    documents = []
    for item in company_data:
        doc = Document(
            content=item["content"],
            meta={
                "title": item["title"],
                "category": item["category"],
                "last_updated": item["last_updated"],
                "source": "company_kb"
            }
        )
        documents.append(doc)
    
    return documents

# Generate the knowledge base
company_docs = generate_company_knowledge_base()
print(f"✅ Generated {len(company_docs)} company documents")
print("\nSample document:")
print(f"Title: {company_docs[0].meta['title']}")
print(f"Content: {company_docs[0].content[:100]}...")

✅ Generated 8 company documents

Sample document:
Title: Remote Work Policy
Content: TechCorp allows employees to work remotely up to 3 days per week. 
            Remote work requests ...


## 5. Example 1: Simple Q&A System

Let's build a complete RAG pipeline for answering questions about company policies.

### 5.1 Create Pipeline Components

In [10]:
# Initialize document store and add documents
company_store = InMemoryDocumentStore()
company_store.write_documents(company_docs)

print(f"📚 Document store contains {company_store.count_documents()} documents")

📚 Document store contains 8 documents


In [11]:
# Define the RAG prompt template
rag_template = """
You are a helpful HR assistant for TechCorp. Answer the employee's question based on the company policies provided.

Company Policies:
{% for doc in documents %}
---
Policy: {{ doc.meta.title }}
{{ doc.content }}
{% endfor %}

Employee Question: {{ query }}

Instructions:
- Provide a clear, concise answer based on the policies above
- If the information isn't in the policies, say so honestly
- Cite the specific policy title when applicable
- Be friendly and professional

Answer:
"""

### 5.2 Build the RAG Pipeline

Haystack uses a Pipeline to connect components together.

In [12]:
# Create pipeline components
retriever = InMemoryBM25Retriever(document_store=company_store)
prompt_builder = PromptBuilder(template=rag_template)
llm = OpenAIGenerator(
    api_key=Secret.from_env_var("OPENAI_API_KEY"),
    model="gpt-3.5-turbo",
    generation_kwargs={"temperature": 0.2}  # Lower temperature for factual responses
)

# Build the pipeline
rag_pipeline = Pipeline()
rag_pipeline.add_component("retriever", retriever)
rag_pipeline.add_component("prompt_builder", prompt_builder)
rag_pipeline.add_component("llm", llm)

# Connect the components
rag_pipeline.connect("retriever.documents", "prompt_builder.documents")
rag_pipeline.connect("prompt_builder", "llm")

print("✅ RAG Pipeline created successfully!")

# Visualize the pipeline
print("\nPipeline structure:")
print(rag_pipeline)

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


✅ RAG Pipeline created successfully!

Pipeline structure:
🚅 Components
  - retriever: InMemoryBM25Retriever
  - prompt_builder: PromptBuilder
  - llm: OpenAIGenerator
🛤️ Connections
  - retriever.documents -> prompt_builder.documents (list[Document])
  - prompt_builder.prompt -> llm.prompt (str)



### 5.3 Test the Pipeline

In [13]:
def ask_question(pipeline: Pipeline, question: str, top_k: int = 3) -> Dict[str, Any]:
      """
      Ask a question using the RAG pipeline.
      """
      start_time = time.time()

      result = pipeline.run({
          "retriever": {"query": question, "top_k": top_k},
          "prompt_builder": {"query": question}
      },
      include_outputs_from={"retriever", "prompt_builder", "llm"})

      end_time = time.time()

      # Debug: Print what keys are actually in the result
      print(f"DEBUG - Result keys: {result.keys()}")

      # Check if retriever output exists
      if "retriever" in result:
          retrieved_docs = result["retriever"]["documents"]
      else:
          print("WARNING: No retriever output found!")
          retrieved_docs = []

      return {
          "question": question,
          "answer": result["llm"]["replies"][0],
          "retrieved_docs": retrieved_docs,
          "response_time": end_time - start_time
      }

In [14]:
# def ask_question(pipeline: Pipeline, question: str, top_k: int = 3) -> Dict[str, Any]:
#     """
#     Ask a question using the RAG pipeline.
    
#     Args:
#         pipeline: The RAG pipeline
#         question: User's question
#         top_k: Number of documents to retrieve
        
#     Returns:
#         Dictionary containing answer and metadata
#     """
#     start_time = time.time()
    
#     result = pipeline.run({
#         "retriever": {"query": question, "top_k": top_k},
#         "prompt_builder": {"query": question}
#     })
    
#     end_time = time.time()
    
#     return {
#         "question": question,
#         "answer": result["llm"]["replies"][0],
#         "retrieved_docs": result["retriever"]["documents"],
#         "response_time": end_time - start_time
#     }

# Test with sample questions
test_questions = [
    "How many vacation days do I get per year?",
    "Can I work from home?",
    "What's the health insurance coverage?"
]

for question in test_questions:
    print("\n" + "="*80)
    print(f"❓ Question: {question}")
    print("="*80)
    
    result = ask_question(rag_pipeline, question)
    
    print(f"\n💡 Answer:\n{result['answer']}")
    print(f"\n⏱️  Response time: {result['response_time']:.2f}s")
    print(f"\n📄 Retrieved {len(result['retrieved_docs'])} relevant policies:")
    for i, doc in enumerate(result['retrieved_docs'], 1):
        print(f"   {i}. {doc.meta['title']} (score: {doc.score:.3f})")


❓ Question: How many vacation days do I get per year?
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

💡 Answer:
Hi there! According to our Vacation Policy, full-time employees at TechCorp receive 20 days of paid vacation annually. Vacation days accrue monthly at a rate of 1.67 days per month. If you have any more questions or need further clarification, feel free to reach out. Have a great day!

⏱️  Response time: 1.55s

📄 Retrieved 3 relevant policies:
   1. Vacation Policy (score: 8.576)
   2. Remote Work Policy (score: 4.856)
   3. Health Insurance (score: 4.193)

❓ Question: Can I work from home?
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

💡 Answer:
Yes, you can work from home up to 3 days per week as per our Remote Work Policy. Please ensure to get approval from your direct manager and be available during core hours (10 AM - 3 PM local time). The company will provide the necessary equipment for remote work.

⏱️  Response time: 1

## 6. Example 2: Technical Documentation Assistant

Let's build a RAG system for answering Python programming questions using documentation.

In [15]:
def generate_python_docs() -> List[Document]:
    """
    Generate sample Python documentation snippets.
    """
    python_docs = [
        {
            "title": "List Comprehensions",
            "content": """List comprehensions provide a concise way to create lists. 
            Common applications are to make new lists where each element is the result of some 
            operations applied to each member of another sequence. 
            Syntax: [expression for item in iterable if condition]
            Example: squares = [x**2 for x in range(10)]""",
            "topic": "data_structures"
        },
        {
            "title": "Dictionaries",
            "content": """Dictionaries are unordered collections of key-value pairs. 
            Keys must be immutable (strings, numbers, tuples). Values can be any type.
            Creation: my_dict = {'key': 'value', 'number': 42}
            Access: my_dict['key'] or my_dict.get('key')
            Methods: keys(), values(), items(), update(), pop()""",
            "topic": "data_structures"
        },
        {
            "title": "Exception Handling",
            "content": """Use try-except blocks to handle exceptions gracefully.
            Syntax: try: risky_operation() except ExceptionType as e: handle_error(e)
            Multiple exceptions: except (TypeError, ValueError) as e:
            Finally block: Executes regardless of exception. Use for cleanup.
            Raise custom exceptions: raise ValueError('Invalid input')""",
            "topic": "error_handling"
        },
        {
            "title": "File I/O",
            "content": """Python provides built-in functions for file operations.
            Open file: with open('file.txt', 'r') as f: content = f.read()
            Modes: 'r' (read), 'w' (write), 'a' (append), 'rb' (read binary)
            Methods: read(), readline(), readlines(), write(), writelines()
            Context manager (with) automatically closes files.""",
            "topic": "io"
        },
        {
            "title": "Lambda Functions",
            "content": """Lambda functions are small anonymous functions defined with lambda keyword.
            Syntax: lambda arguments: expression
            Example: square = lambda x: x**2
            Common with map, filter, sorted: sorted(items, key=lambda x: x['value'])
            Limited to single expression, no statements.""",
            "topic": "functions"
        },
        {
            "title": "Generators",
            "content": """Generators are functions that return iterators using yield.
            Memory efficient for large sequences - values generated on-the-fly.
            Example: def count_up_to(n): for i in range(n): yield i
            Generator expressions: (x**2 for x in range(10))
            Use next() to get values or iterate in for loop.""",
            "topic": "iterators"
        }
    ]
    
    return [
        Document(
            content=doc["content"],
            meta={"title": doc["title"], "topic": doc["topic"], "source": "python_docs"}
        )
        for doc in python_docs
    ]

# Generate Python documentation
python_docs = generate_python_docs()
print(f"✅ Generated {len(python_docs)} Python documentation snippets")

✅ Generated 6 Python documentation snippets


In [16]:
# Create a new pipeline for Python documentation
python_store = InMemoryDocumentStore()
python_store.write_documents(python_docs)

# Template optimized for code documentation
code_template = """
You are a Python programming assistant. Help the user understand Python concepts based on the documentation.

Documentation:
{% for doc in documents %}
## {{ doc.meta.title }}
{{ doc.content }}

{% endfor %}

Question: {{ query }}

Provide a clear explanation with code examples where applicable.
Answer:
"""

# Build Python docs pipeline
python_pipeline = Pipeline()
python_pipeline.add_component("retriever", InMemoryBM25Retriever(document_store=python_store))
python_pipeline.add_component("prompt_builder", PromptBuilder(template=code_template))
python_pipeline.add_component("llm", OpenAIGenerator(
    api_key=Secret.from_env_var("OPENAI_API_KEY"),
    model="gpt-3.5-turbo"
))

python_pipeline.connect("retriever.documents", "prompt_builder.documents")
python_pipeline.connect("prompt_builder", "llm")

print("✅ Python documentation pipeline ready!")

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


✅ Python documentation pipeline ready!


In [17]:
# Test Python documentation assistant
python_questions = [
    "How do I handle errors in Python?",
    "What's the difference between a list comprehension and a generator?",
    "How do I read a file safely?"
]

for question in python_questions:
    print("\n" + "="*80)
    print(f"❓ {question}")
    print("="*80)
    
    result = ask_question(python_pipeline, question, top_k=2)
    print(f"\n{result['answer']}")


❓ How do I handle errors in Python?
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

In Python, you can handle errors using try-except blocks. This allows you to gracefully manage exceptions and prevent your program from crashing. 

Here is an example of how you can use try-except blocks to handle errors:

```python
try:
    num1 = int(input("Enter a number: "))
    num2 = int(input("Enter another number: "))
    
    result = num1 / num2
    print("Result:", result)
    
except ZeroDivisionError as e:
    print("Error: Cannot divide by zero")
    
except ValueError as e:
    print("Error: Invalid input. Please enter a valid number.")
```

In this code snippet, we are attempting to divide two numbers input by the user. If a ZeroDivisionError or ValueError occurs during the execution of the code inside the try block, the corresponding except block will handle the error. 

You can also use a finally block to execute code regardless of whether an exception occurred

## 7. Example 3: News Article Summarization with Citations

Let's build a system that answers questions about news articles with proper source citations.

In [18]:
def generate_news_articles() -> List[Document]:
    """
    Generate sample news articles.
    """
    news_data = [
        {
            "title": "Major Breakthrough in Quantum Computing",
            "content": """Scientists at Tech University announced a significant breakthrough in 
            quantum computing, achieving quantum supremacy with a 1000-qubit processor. The new 
            system can solve complex optimization problems 1000x faster than classical supercomputers. 
            Lead researcher Dr. Smith states this could revolutionize drug discovery and climate modeling.""",
            "category": "Technology",
            "date": "2025-01-28",
            "source_url": "https://technews.example.com/quantum-breakthrough"
        },
        {
            "title": "New Climate Agreement Reached",
            "content": """World leaders agreed to reduce carbon emissions by 50% by 2035 at the 
            Global Climate Summit. The agreement includes $500 billion in funding for renewable energy 
            in developing nations. Environmental groups praised the commitment while noting challenges 
            in implementation.""",
            "category": "Environment",
            "date": "2025-01-27",
            "source_url": "https://worldnews.example.com/climate-agreement"
        },
        {
            "title": "AI Assists in Medical Diagnosis",
            "content": """A new AI system developed by MedTech Corp can diagnose rare diseases 
            with 95% accuracy from medical imaging. The system has been trained on 10 million medical 
            images and can identify patterns invisible to human doctors. Clinical trials show 30% 
            improvement in early detection rates.""",
            "category": "Healthcare",
            "date": "2025-01-26",
            "source_url": "https://healthnews.example.com/ai-diagnosis"
        },
        {
            "title": "Electric Vehicle Sales Surge",
            "content": """Electric vehicle sales increased 200% year-over-year, now comprising 
            25% of all new car sales globally. Major automakers announced plans to phase out combustion 
            engines by 2030. Battery technology improvements have extended range to 500+ miles while 
            reducing costs by 40%.""",
            "category": "Automotive",
            "date": "2025-01-25",
            "source_url": "https://autonews.example.com/ev-sales-surge"
        }
    ]
    
    return [
        Document(
            content=doc["content"],
            meta={
                "title": doc["title"],
                "category": doc["category"],
                "date": doc["date"],
                "source_url": doc["source_url"],
                "source": "news"
            }
        )
        for doc in news_data
    ]

news_docs = generate_news_articles()
print(f"✅ Generated {len(news_docs)} news articles")

✅ Generated 4 news articles


In [19]:
# Template with citation requirements
news_template = """
You are a news assistant. Answer questions based on the news articles provided.

News Articles:
{% for doc in documents %}
---
Title: {{ doc.meta.title }}
Date: {{ doc.meta.date }}
Source: {{ doc.meta.source_url }}
Content: {{ doc.content }}
{% endfor %}

Question: {{ query }}

Instructions:
- Answer based on the articles provided
- Include citations with article titles and dates
- If information is not in the articles, state that clearly
- Provide source URLs when relevant

Answer:
"""

# Build news pipeline
news_store = InMemoryDocumentStore()
news_store.write_documents(news_docs)

news_pipeline = Pipeline()
news_pipeline.add_component("retriever", InMemoryBM25Retriever(document_store=news_store))
news_pipeline.add_component("prompt_builder", PromptBuilder(template=news_template))
news_pipeline.add_component("llm", OpenAIGenerator(
    api_key=Secret.from_env_var("OPENAI_API_KEY"),
    model="gpt-3.5-turbo"
))

news_pipeline.connect("retriever.documents", "prompt_builder.documents")
news_pipeline.connect("prompt_builder", "llm")

print("✅ News pipeline ready!")

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


✅ News pipeline ready!


In [20]:
# Test with news questions
news_questions = [
    "What are the recent developments in AI for healthcare?",
    "What progress has been made in quantum computing?",
    "What's happening with electric vehicles?"
]

for question in news_questions:
    print("\n" + "="*80)
    print(f"❓ {question}")
    print("="*80)
    
    result = ask_question(news_pipeline, question, top_k=2)
    print(f"\n{result['answer']}")
    
    print("\n📰 Sources:")
    for doc in result['retrieved_docs']:
        print(f"  • {doc.meta['title']} ({doc.meta['date']})")
        print(f"    {doc.meta['source_url']}")


❓ What are the recent developments in AI for healthcare?
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

The recent development in AI for healthcare is a new AI system developed by MedTech Corp, as reported in the article titled "AI Assists in Medical Diagnosis" on January 26, 2025. This AI system can diagnose rare diseases with 95% accuracy from medical imaging, thanks to being trained on 10 million medical images. Clinical trials have shown a 30% improvement in early detection rates due to this AI system. For more information, you can refer to the source: https://healthnews.example.com/ai-diagnosis. 

There is no information on other recent developments in AI for healthcare in the provided articles.

📰 Sources:
  • New Climate Agreement Reached (2025-01-27)
    https://worldnews.example.com/climate-agreement
  • AI Assists in Medical Diagnosis (2025-01-26)
    https://healthnews.example.com/ai-diagnosis

❓ What progress has been made in quantum computing?
DEB

## 8. Prompt Engineering Best Practices

Let's explore how different prompt structures affect RAG performance.

In [21]:
# Different prompt templates to compare
prompt_variations = {
    "basic": """
Context: {% for doc in documents %}{{ doc.content }}{% endfor %}
Question: {{ query }}
Answer:
""",
    
    "structured": """
You are a helpful assistant. Use the context below to answer the question.

CONTEXT:
{% for doc in documents %}
[{{ loop.index }}] {{ doc.content }}
{% endfor %}

QUESTION: {{ query }}

ANSWER:
""",
    
    "with_instructions": """
You are a precise assistant. Follow these guidelines:
1. Base your answer ONLY on the context provided
2. If the context doesn't contain the answer, say "I don't have enough information"
3. Be concise but complete
4. Cite specific parts of the context when possible

CONTEXT:
{% for doc in documents %}
Source {{ loop.index }}: {{ doc.content }}
{% endfor %}

USER QUESTION: {{ query }}

YOUR ANSWER:
""",
    
    "chain_of_thought": """
You are a thoughtful assistant. Answer the question step-by-step.

CONTEXT:
{% for doc in documents %}
{{ doc.content }}
{% endfor %}

QUESTION: {{ query }}

Let's think through this step by step:
1. First, identify relevant information from the context
2. Then, synthesize the information
3. Finally, provide a clear answer

ANSWER:
"""
}

print("✅ Created 4 prompt variations for comparison")

✅ Created 4 prompt variations for comparison


In [22]:
def compare_prompts(question: str, store: InMemoryDocumentStore, templates: Dict[str, str]):
    """
    Compare different prompt templates for the same question.
    """
    print(f"\n{'='*80}")
    print(f"Comparing prompts for: {question}")
    print(f"{'='*80}\n")
    
    for name, template in templates.items():
        # Build temporary pipeline with this template
        temp_pipeline = Pipeline()
        temp_pipeline.add_component("retriever", InMemoryBM25Retriever(document_store=store))
        temp_pipeline.add_component("prompt_builder", PromptBuilder(template=template))
        temp_pipeline.add_component("llm", OpenAIGenerator(
            api_key=Secret.from_env_var("OPENAI_API_KEY"),
            model="gpt-3.5-turbo"
        ))
        temp_pipeline.connect("retriever.documents", "prompt_builder.documents")
        temp_pipeline.connect("prompt_builder", "llm")
        
        result = ask_question(temp_pipeline, question, top_k=2)
        
        print(f"\n📝 Template: {name.upper()}")
        print("-" * 80)
        print(result['answer'])
        print(f"\n⏱️ Response time: {result['response_time']:.2f}s")

# Compare prompts
compare_prompts(
    "How many vacation days do employees get?",
    company_store,
    prompt_variations
)

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.



Comparing prompts for: How many vacation days do employees get?



PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

📝 Template: BASIC
--------------------------------------------------------------------------------
Employees get 20 days of paid vacation annually.

⏱️ Response time: 0.62s


PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

📝 Template: STRUCTURED
--------------------------------------------------------------------------------
Full-time employees receive 20 days of paid vacation annually.

⏱️ Response time: 0.63s


PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

📝 Template: WITH_INSTRUCTIONS
--------------------------------------------------------------------------------
Full-time employees receive 20 days of paid vacation annually, which accrue at a rate of 1.67 days per month. Unused vacation days can be carried over up to 5 days into the next year.

⏱️ Response time: 0.87s
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

📝 Template: CHAIN_OF_THOUGHT
--------------------------------------------------------------------------------
1. Relevant information from the context:
- Full-time employees receive 20 days of paid vacation annually, accruing 1.67 days per month.
- Unused vacation days can be carried over up to 5 days into the next year.

2. Synthesizing the information:
- Employees receive 20 days of paid vacation annually.
- This means they accrue approximately 1.67 days of vacation per month.
- They can carry over up to 5 unused vacation days i

### Key Prompt Engineering Insights

1. **Clear Instructions**: Explicit guidelines improve answer quality
2. **Structure**: Well-formatted context helps LLM parse information
3. **Constraints**: Tell the model what NOT to do (e.g., don't make up info)
4. **Chain of Thought**: Asking for step-by-step reasoning can improve accuracy
5. **Citation Requirements**: Explicitly ask for sources if needed

## 9. Basic Evaluation

Let's implement simple evaluation metrics for our RAG system.

In [23]:
def evaluate_retrieval_precision(pipeline: Pipeline, test_cases: List[Dict[str, Any]]):
    """
    Evaluate retrieval precision - are retrieved documents relevant?
    
    Args:
        pipeline: RAG pipeline to evaluate
        test_cases: List of dicts with 'query' and 'expected_doc_titles'
    """
    results = []
    
    for case in test_cases:
        query = case['query']
        expected_titles = set(case['expected_doc_titles'])
        
        result = ask_question(pipeline, query, top_k=3)
        retrieved_titles = {doc.meta['title'] for doc in result['retrieved_docs']}
        
        # Calculate precision: relevant retrieved / total retrieved
        relevant_retrieved = retrieved_titles.intersection(expected_titles)
        precision = len(relevant_retrieved) / len(retrieved_titles) if retrieved_titles else 0
        
        results.append({
            'query': query,
            'precision': precision,
            'retrieved': list(retrieved_titles),
            'expected': list(expected_titles),
            'relevant_found': list(relevant_retrieved)
        })
    
    # Calculate average precision
    avg_precision = np.mean([r['precision'] for r in results])
    
    return results, avg_precision

# Define test cases
test_cases = [
    {
        'query': 'How many vacation days do I get?',
        'expected_doc_titles': ['Vacation Policy']
    },
    {
        'query': 'Can I work remotely?',
        'expected_doc_titles': ['Remote Work Policy']
    },
    {
        'query': 'What health benefits are available?',
        'expected_doc_titles': ['Health Insurance']
    },
    {
        'query': 'How does performance review work?',
        'expected_doc_titles': ['Performance Reviews']
    }
]

# Evaluate
results, avg_precision = evaluate_retrieval_precision(rag_pipeline, test_cases)

print("\n" + "="*80)
print("RETRIEVAL PRECISION EVALUATION")
print("="*80)

for result in results:
    print(f"\nQuery: {result['query']}")
    print(f"Precision: {result['precision']:.2%}")
    print(f"Expected: {result['expected']}")
    print(f"Retrieved: {result['retrieved']}")
    print(f"Relevant found: {result['relevant_found']}")

print(f"\n{'='*80}")
print(f"Average Precision: {avg_precision:.2%}")
print(f"{'='*80}")

DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

RETRIEVAL PRECISION EVALUATION

Query: How many vacation days do I get?
Precision: 33.33%
Expected: ['Vacation Policy']
Retrieved: ['Health Insurance', 'Vacation Policy', 'Remote Work Policy']
Relevant found: ['Vacation Policy']

Query: Can I work remotely?
Precision: 33.33%
Expected: ['Remote Work Policy']
Retrieved: ['Equipment and Technology', 'Vacation Policy', 'Remote Work Policy']
Relevant found: ['Remote Work Policy']

Query: What health benefits are available?
Precision: 33.33%
Expected: ['Health Insurance']
Retrieved: ['Performance Reviews', 'Health Insurance', 'Remote Work Policy']
Relevant found: ['Health Insurance']

Query: How does performance review work?
Precision: 33.33%
Expected: ['Perform

In [24]:
def measure_response_times(pipeline: Pipeline, queries: List[str], num_runs: int = 3):
    """
    Measure and analyze response times.
    """
    times = []
    
    for query in queries:
        query_times = []
        for _ in range(num_runs):
            result = ask_question(pipeline, query)
            query_times.append(result['response_time'])
        times.append({
            'query': query,
            'mean': np.mean(query_times),
            'std': np.std(query_times),
            'min': np.min(query_times),
            'max': np.max(query_times)
        })
    
    return times

# Measure response times
timing_queries = [
    "How many vacation days?",
    "What's the remote work policy?",
    "Health insurance coverage?"
]

timings = measure_response_times(rag_pipeline, timing_queries, num_runs=2)

print("\n" + "="*80)
print("RESPONSE TIME ANALYSIS")
print("="*80)

timing_df = pd.DataFrame(timings)
print("\n", timing_df.to_string(index=False))

print(f"\nOverall Average: {timing_df['mean'].mean():.2f}s")
print(f"Overall Std Dev: {timing_df['std'].mean():.2f}s")

DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

RESPONSE TIME ANALYSIS

                          query     mean      std      min      max
       How many vacation days? 1.236576 0.223129 1.013448 1.459705
What's the remote work policy? 1.012922 0.064294 0.948627 1.077216
    Health insurance coverage? 1.466646 0.348042 1.118604 1.814687

Overall Average: 1.24s
Overall Std Dev: 0.21s


## 10. Exercises & Challenges

Try these exercises to reinforce your learning:

### Exercise 1: Add More Documents

Extend the company knowledge base with 3-5 additional policies:
- Parental leave policy
- Referral bonus program
- Office amenities
- Relocation assistance

Test if the RAG system retrieves them correctly.

In [25]:
### Exercise 1: Add More Documents
# Create new sample documents and add to document store
# Extend the company knowledge base with 3-5 additional policies

# Define new company policies
new_policies = [
    {
        "title": "Parental Leave Policy",
        "content": """TechCorp provides 16 weeks of paid parental leave for primary caregivers 
        and 8 weeks for secondary caregivers. Leave can be taken within the first year 
        after birth or adoption. Employees can choose to take leave continuously or split 
        it into multiple periods with manager approval. Benefits continue during parental leave.""",
        "category": "HR",
        "last_updated": "2025-01-22"
    },
    {
        "title": "Referral Bonus Program",
        "content": """Employees receive a $3,000 bonus for successful referrals of qualified candidates. 
        The bonus is paid in two installments: $1,000 after the new hire completes 90 days, 
        and $2,000 after 6 months. Referrals must be submitted through the careers portal 
        before the candidate applies. Managers cannot refer candidates for their own teams.""",
        "category": "HR",
        "last_updated": "2025-01-19"
    },
    {
        "title": "Office Amenities",
        "content": """TechCorp offices feature free snacks and beverages, including coffee, tea, 
        and fresh fruit. Lunch is provided on Wednesdays and Fridays. Facilities include 
        a gym with shower facilities, game rooms, quiet focus rooms, and mother's rooms. 
        Employees have access to on-site bike storage and electric vehicle charging stations.""",
        "category": "Benefits",
        "last_updated": "2025-01-14"
    },
    {
        "title": "Relocation Assistance",
        "content": """TechCorp provides comprehensive relocation assistance for eligible positions. 
        The package includes up to $10,000 for moving expenses, temporary housing for 30 days, 
        and two house-hunting trips. Employees must remain with the company for 24 months 
        or repay relocation costs on a prorated basis. Assistance is available for domestic 
        and international relocations.""",
        "category": "Benefits",
        "last_updated": "2025-01-11"
    },
    {
        "title": "Flexible Work Hours",
        "content": """TechCorp offers flexible work hours with core collaboration hours from 10 AM to 3 PM. 
        Employees can adjust their start and end times to accommodate personal needs. 
        Teams coordinate schedules to ensure adequate coverage. Flexible hours must be 
        discussed with managers and documented in the HR system.""",
        "category": "HR",
        "last_updated": "2025-01-16"
    }
]

# Convert to Document objects
new_documents = [
    Document(
        content=policy["content"],
        meta={
            "title": policy["title"],
            "category": policy["category"],
            "last_updated": policy["last_updated"],
            "source": "company_kb"
        }
    )
    for policy in new_policies
]

# Add to the existing document store
company_store.write_documents(new_documents)

print(f"✅ Added {len(new_documents)} new policies to the document store")
print(f"📚 Total documents in store: {company_store.count_documents()}")
print("\nNew policies added:")
for policy in new_policies:
    print(f"  • {policy['title']}")

# Test if the RAG system retrieves them correctly by using relevant queries
print("\n" + "="*80)
print("TESTING NEW POLICIES WITH RAG PIPELINE")
print("="*80)

test_queries = [
    "What is the parental leave policy?",
    "How much is the referral bonus?",
    "What amenities are available in the office?",
    "Does the company help with relocation?",
    "Can I have flexible work hours?"
]

for query in test_queries:
    print(f"\n{'='*80}")
    print(f"❓ Query: {query}")
    print(f"{'='*80}")
    
    result = ask_question(rag_pipeline, query, top_k=2)
    
    print(f"\n💡 Answer:\n{result['answer']}")
    print(f"\n⏱️  Response time: {result['response_time']:.2f}s")
    print(f"\n📄 Retrieved documents:")
    for i, doc in enumerate(result['retrieved_docs'], 1):
        print(f"   {i}. {doc.meta['title']} (score: {doc.score:.3f})")

✅ Added 5 new policies to the document store
📚 Total documents in store: 13

New policies added:
  • Parental Leave Policy
  • Referral Bonus Program
  • Office Amenities
  • Relocation Assistance
  • Flexible Work Hours

TESTING NEW POLICIES WITH RAG PIPELINE

❓ Query: What is the parental leave policy?
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

💡 Answer:
TechCorp provides 16 weeks of paid parental leave for primary caregivers and 8 weeks for secondary caregivers. Leave can be taken within the first year after birth or adoption, and employees can choose to take it continuously or split it into multiple periods with manager approval. Benefits continue during parental leave. (Parental Leave Policy)

⏱️  Response time: 1.18s

📄 Retrieved documents:
   1. Parental Leave Policy (score: 7.492)
   2. Security Protocols (score: 4.620)

❓ Query: How much is the referral bonus?
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

💡 Answer:
The ref

### Exercise 2: Improve Prompt Template

Create a prompt template that:
1. Asks the LLM to rate its confidence (1-10)
2. Provides both a short answer and detailed explanation
3. Lists which documents were most useful

Compare results with the basic template.

In [26]:
### Exercise 2: Improve Prompt Template

# Design improved template with confidence rating, short/detailed answers, and document citations
improved_template = """
You are a helpful HR assistant for TechCorp. Answer the employee's question based on the company policies provided.

Company Policies:
{% for doc in documents %}
---
[Document {{ loop.index }}] Policy: {{ doc.meta.title }}
{{ doc.content }}
{% endfor %}

Employee Question: {{ query }}

Instructions:
- Provide your answer in the following structured format:
- Rate your confidence in the answer from 1-10 (10 being most confident)
- Give a short answer (1-2 sentences)
- Provide a detailed explanation with all relevant information
- List which documents were most useful for answering the question

Response Format:
**Confidence:** [1-10]

**Short Answer:**
[Concise 1-2 sentence answer]

**Detailed Explanation:**
[Comprehensive explanation with all relevant details]

**Sources Used:**
[List the document numbers and titles that were most relevant]

Answer:
"""

# Create pipeline with improved template
print("Creating improved RAG pipeline...\n")

improved_prompt_builder = PromptBuilder(template=improved_template)
improved_llm = OpenAIGenerator(
    api_key=Secret.from_env_var("OPENAI_API_KEY"),
    model="gpt-3.5-turbo",
    generation_kwargs={"temperature": 0.2}
)

improved_pipeline = Pipeline()
improved_pipeline.add_component("retriever", InMemoryBM25Retriever(document_store=company_store))
improved_pipeline.add_component("prompt_builder", improved_prompt_builder)
improved_pipeline.add_component("llm", improved_llm)

improved_pipeline.connect("retriever.documents", "prompt_builder.documents")
improved_pipeline.connect("prompt_builder", "llm")

print("✅ Improved pipeline created!\n")

# Compare with basic pipeline
print("="*80)
print("COMPARING BASIC vs IMPROVED PROMPT TEMPLATES")
print("="*80)

comparison_questions = [
    "How many vacation days do I get?",
    "What is the parental leave policy?",
    "Can I get reimbursed for professional development?"
]

for question in comparison_questions:
    print(f"\n{'='*80}")
    print(f"❓ Question: {question}")
    print(f"{'='*80}")
    
    # Get answer from basic pipeline
    print("\n📝 BASIC TEMPLATE RESPONSE:")
    print("-" * 80)
    basic_result = ask_question(rag_pipeline, question, top_k=3)
    print(basic_result['answer'])
    print(f"\n⏱️  Response time: {basic_result['response_time']:.2f}s")
    
    # Get answer from improved pipeline
    print("\n📝 IMPROVED TEMPLATE RESPONSE:")
    print("-" * 80)
    improved_result = ask_question(improved_pipeline, question, top_k=3)
    print(improved_result['answer'])
    print(f"\n⏱️  Response time: {improved_result['response_time']:.2f}s")
    
    print("\n" + "="*80)

# Summary of improvements
print("\n" + "="*80)
print("SUMMARY OF IMPROVEMENTS")
print("="*80)
print("""
The improved template provides:

1. ✅ Confidence Rating (1-10): Helps users understand how certain the answer is
2. ✅ Short Answer: Quick information for users who need immediate answers
3. ✅ Detailed Explanation: Comprehensive information for users who want more context
4. ✅ Source Citations: Transparency about which documents were used

Benefits:
- Better user experience with structured responses
- Increased transparency and trust
- Easier to verify information against source documents
- Confidence scores help identify when answers might be uncertain
""")

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


Creating improved RAG pipeline...

✅ Improved pipeline created!

COMPARING BASIC vs IMPROVED PROMPT TEMPLATES

❓ Question: How many vacation days do I get?

📝 BASIC TEMPLATE RESPONSE:
--------------------------------------------------------------------------------
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])
As a full-time employee at TechCorp, you are entitled to 20 days of paid vacation annually. Vacation days accrue monthly at a rate of 1.67 days per month. You can carry over up to 5 unused vacation days into the next year. (Vacation Policy)

⏱️  Response time: 1.28s

📝 IMPROVED TEMPLATE RESPONSE:
--------------------------------------------------------------------------------
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])
**Confidence:** 10

**Short Answer:**
Full-time employees at TechCorp receive 20 days of paid vacation annually.

**Detailed Explanation:**
According to Document 1 (Vacation Policy), full-time employees at TechCorp

### Exercise 3: Handle Out-of-Scope Questions

Test the pipeline with questions that cannot be answered from the knowledge base:
- "What's the weather today?"
- "How do I cook pasta?"
- "What's the stock price?"

Does the system appropriately say it doesn't know? How can you improve this?

In [27]:
### Exercise 3: Handle Out-of-Scope Questions

# Test with out-of-scope questions
print("="*80)
print("TESTING OUT-OF-SCOPE QUESTIONS WITH BASIC PIPELINE")
print("="*80)

out_of_scope_questions = [
    "What's the weather today?",
    "How do I cook pasta?",
    "What's the stock price?",
    "Who won the Super Bowl?",
    "What is the capital of France?"
]

print("\n🔍 Testing basic pipeline with out-of-scope questions...\n")

for question in out_of_scope_questions:
    print(f"\n{'='*80}")
    print(f"❓ Question: {question}")
    print(f"{'='*80}")
    
    result = ask_question(rag_pipeline, question, top_k=2)
    print(f"\n💡 Answer:\n{result['answer']}")
    print(f"\n📄 Retrieved documents:")
    for i, doc in enumerate(result['retrieved_docs'], 1):
        print(f"   {i}. {doc.meta['title']} (score: {doc.score:.3f})")

# Analyze responses
print("\n\n" + "="*80)
print("ANALYSIS OF BASIC PIPELINE BEHAVIOR")
print("="*80)
print("""
⚠️  ISSUES IDENTIFIED:

1. The system may attempt to answer questions using irrelevant documents
2. It might hallucinate answers based on general knowledge
3. No clear indication when information is not in the knowledge base
4. Retrieved documents have low relevance scores but are still used

💡 IMPROVEMENTS NEEDED:

1. Add explicit instructions to only answer from provided context
2. Require the system to state when information is not available
3. Add relevance score checking
4. Provide a standard "I don't know" response format
""")

# Modify prompt to handle better
print("\n" + "="*80)
print("CREATING IMPROVED PIPELINE FOR OUT-OF-SCOPE HANDLING")
print("="*80)

improved_oos_template = """
You are a helpful HR assistant for TechCorp. Your role is to answer questions ONLY about TechCorp company policies.

Company Policies:
{% for doc in documents %}
---
Policy: {{ doc.meta.title }}
{{ doc.content }}
{% endfor %}

Employee Question: {{ query }}

CRITICAL INSTRUCTIONS:
1. ONLY answer if the information is explicitly present in the company policies above
2. If the question is about topics outside of TechCorp policies (weather, cooking, stocks, sports, general knowledge, etc.), respond with:
   "I apologize, but I can only answer questions about TechCorp company policies. Your question about [topic] is outside my knowledge base. Please ask about our policies regarding vacation, benefits, remote work, or other company-related topics."
3. If the question is about TechCorp but the specific information is not in the policies, respond with:
   "I don't have information about that specific topic in our current policy documents. Please contact HR directly at hr@techcorp.com for assistance."
4. Do NOT make up information or use general knowledge
5. Do NOT try to be helpful by answering questions outside your scope

Answer:
"""

# Create improved pipeline
improved_oos_prompt_builder = PromptBuilder(template=improved_oos_template)
improved_oos_llm = OpenAIGenerator(
    api_key=Secret.from_env_var("OPENAI_API_KEY"),
    model="gpt-3.5-turbo",
    generation_kwargs={"temperature": 0.1}  # Lower temperature for more consistent behavior
)

improved_oos_pipeline = Pipeline()
improved_oos_pipeline.add_component("retriever", InMemoryBM25Retriever(document_store=company_store))
improved_oos_pipeline.add_component("prompt_builder", improved_oos_prompt_builder)
improved_oos_pipeline.add_component("llm", improved_oos_llm)

improved_oos_pipeline.connect("retriever.documents", "prompt_builder.documents")
improved_oos_pipeline.connect("prompt_builder", "llm")

print("\n✅ Improved out-of-scope handling pipeline created!\n")

# Test improved pipeline
print("="*80)
print("TESTING IMPROVED PIPELINE WITH OUT-OF-SCOPE QUESTIONS")
print("="*80)

for question in out_of_scope_questions:
    print(f"\n{'='*80}")
    print(f"❓ Question: {question}")
    print(f"{'='*80}")
    
    result = ask_question(improved_oos_pipeline, question, top_k=2)
    print(f"\n💡 Improved Answer:\n{result['answer']}")

# Test with in-scope questions to ensure it still works
print("\n\n" + "="*80)
print("VERIFYING IN-SCOPE QUESTIONS STILL WORK CORRECTLY")
print("="*80)

in_scope_questions = [
    "How many vacation days do I get?",
    "Can I work remotely?"
]

for question in in_scope_questions:
    print(f"\n{'='*80}")
    print(f"❓ Question: {question}")
    print(f"{'='*80}")
    
    result = ask_question(improved_oos_pipeline, question, top_k=2)
    print(f"\n💡 Answer:\n{result['answer']}")

# Summary
print("\n\n" + "="*80)
print("SUMMARY: OUT-OF-SCOPE HANDLING IMPROVEMENTS")
print("="*80)
print("""
✅ IMPROVEMENTS IMPLEMENTED:

1. Explicit scope definition: System knows it's only for TechCorp policies
2. Clear rejection of out-of-scope questions with helpful messaging
3. Lower temperature (0.1) for more consistent boundary enforcement
4. Structured response templates for different scenarios
5. Guidance to contact HR for missing information

📊 RESULTS:

- Out-of-scope questions are now properly rejected
- Users receive clear guidance on what the system can help with
- In-scope questions still receive accurate answers
- Reduced risk of hallucination or incorrect information

🎯 BEST PRACTICES:

- Always define clear system boundaries in prompts
- Provide helpful alternatives when rejecting questions
- Use lower temperature for stricter adherence to instructions
- Test both in-scope and out-of-scope scenarios
- Monitor and iterate based on user feedback
""")

TESTING OUT-OF-SCOPE QUESTIONS WITH BASIC PIPELINE

🔍 Testing basic pipeline with out-of-scope questions...


❓ Question: What's the weather today?
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

💡 Answer:
I'm happy to help with any questions you have related to our company policies. However, the information about the weather is not included in our policies. If you have any other questions regarding our Referral Bonus Program or Health Insurance policies, feel free to ask!

📄 Retrieved documents:
   1. Referral Bonus Program (score: 0.702)
   2. Health Insurance (score: 0.682)

❓ Question: How do I cook pasta?
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

💡 Answer:
I'm sorry, but our company policies do not cover cooking instructions. However, I recommend checking online for easy pasta recipes or asking a friend or family member for guidance. If you have any other work-related questions, feel free to ask!

📄 Retrieved documents:

❓ Ques

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

💡 Answer:
I'm happy to help with any questions related to TechCorp policies. However, the capital of France is not covered in our company policies. If you have any other questions about equipment, technology, or our referral bonus program, feel free to ask!

📄 Retrieved documents:
   1. Equipment and Technology (score: 2.754)
   2. Referral Bonus Program (score: 2.723)


ANALYSIS OF BASIC PIPELINE BEHAVIOR

⚠️  ISSUES IDENTIFIED:

1. The system may attempt to answer questions using irrelevant documents
2. It might hallucinate answers based on general knowledge
3. No clear indication when information is not in the knowledge base
4. Retrieved documents have low relevance scores but are still used

💡 IMPROVEMENTS NEEDED:

1. Add explicit instructions to only answer from provided context
2. Require the system to state when information is not available
3. Add relevance score checking
4. Provide a standard "I don't know"

### Challenge: Build a Multi-Topic RAG System

Combine the company policies, Python documentation, and news articles into a single system that:
1. Automatically determines which knowledge base to search
2. Routes queries appropriately
3. Provides answers with source attribution

Hint: You might need multiple retrievers or metadata filtering.

In [28]:
### Challenge: Build a Multi-Topic RAG System

print("="*80)
print("BUILDING MULTI-TOPIC RAG SYSTEM")
print("="*80)

# Step 1: Combine all documents with enhanced metadata
print("\n📚 Step 1: Combining all knowledge bases...\n")

# Get company documents (already have these)
company_documents = list(company_store.filter_documents())
for doc in company_documents:
    doc.meta['topic'] = 'company_policy'
    doc.meta['domain'] = 'HR and Benefits'

# Get Python documentation (from earlier in notebook)
python_documents = generate_python_docs()
for doc in python_documents:
    doc.meta['topic'] = 'technical_documentation'
    doc.meta['domain'] = 'Python Programming'

# Get news articles (from earlier in notebook)
news_documents = generate_news_articles()
for doc in news_documents:
    doc.meta['topic'] = 'news'
    doc.meta['domain'] = 'Current Events'

# Combine all documents
all_documents = company_documents + python_documents + news_documents

print(f"✅ Combined {len(all_documents)} documents:")
print(f"   - Company Policies: {len(company_documents)}")
print(f"   - Python Documentation: {len(python_documents)}")
print(f"   - News Articles: {len(news_documents)}")

# Create unified document store
unified_store = InMemoryDocumentStore()
unified_store.write_documents(all_documents)

print(f"\n📦 Unified document store created with {unified_store.count_documents()} documents")

# Step 2: Create query routing logic
print("\n🧭 Step 2: Implementing query routing logic...\n")

def classify_query(query: str) -> str:
    """
    Classify the query to determine which knowledge base to search.
    Returns: 'company_policy', 'technical_documentation', or 'news'
    """
    query_lower = query.lower()
    
    # Company policy keywords
    company_keywords = [
        'vacation', 'leave', 'policy', 'benefit', 'insurance', 'remote', 'work from home',
        'salary', 'bonus', 'referral', 'office', 'relocation', 'equipment', 'hr',
        'performance review', 'professional development', 'parental', 'flexible hours'
    ]
    
    # Technical documentation keywords
    tech_keywords = [
        'python', 'code', 'programming', 'function', 'class', 'error', 'exception',
        'list', 'dictionary', 'loop', 'variable', 'import', 'module', 'syntax',
        'debug', 'compile', 'generator', 'decorator', 'comprehension'
    ]
    
    # News keywords
    news_keywords = [
        'news', 'recent', 'latest', 'breakthrough', 'announcement', 'development',
        'quantum', 'climate', 'electric vehicle', 'ai', 'medical', 'technology news',
        'research', 'discovery', 'innovation'
    ]
    
    # Count keyword matches
    company_score = sum(1 for kw in company_keywords if kw in query_lower)
    tech_score = sum(1 for kw in tech_keywords if kw in query_lower)
    news_score = sum(1 for kw in news_keywords if kw in query_lower)
    
    # Determine category
    if company_score > tech_score and company_score > news_score:
        return 'company_policy'
    elif tech_score > company_score and tech_score > news_score:
        return 'technical_documentation'
    elif news_score > 0:
        return 'news'
    else:
        # Default to company policy for ambiguous queries
        return 'company_policy'

print("✅ Query classification function created")

# Step 3: Build unified pipeline with routing
print("\n🔧 Step 3: Building unified pipeline...\n")

# Create a smart prompt template that adapts based on topic
unified_template = """
You are a helpful AI assistant with access to multiple knowledge bases.

Query Topic: {{ topic }}
Domain: {{ domain }}

Relevant Information:
{% for doc in documents %}
---
Source: {{ doc.meta.title if doc.meta.title else 'Document ' + loop.index|string }}
Category: {{ doc.meta.category if doc.meta.category else doc.meta.topic }}
{{ doc.content }}
{% endfor %}

User Question: {{ query }}

Instructions:
- Provide a clear, accurate answer based on the information above
- Cite specific sources when applicable
- If the information is not sufficient, say so honestly
- Adapt your tone based on the topic (professional for company policies, technical for programming, informative for news)

Answer:
"""

def ask_unified_question(query: str, top_k: int = 3) -> Dict[str, Any]:
    """
    Ask a question using the unified multi-topic RAG system.
    """
    start_time = time.time()
    
    # Step 1: Classify the query
    topic = classify_query(query)
    
    # Step 2: Retrieve documents filtered by topic
    retriever = InMemoryBM25Retriever(document_store=unified_store)
    retrieval_result = retriever.run(
        query=query,
        top_k=top_k,
        filters={"field": "topic", "operator": "==", "value": topic}
    )
    
    retrieved_docs = retrieval_result['documents']
    
    # Determine domain from retrieved docs
    domain = retrieved_docs[0].meta.get('domain', 'General') if retrieved_docs else 'General'
    
    # Step 3: Build prompt with topic context
    prompt_builder = PromptBuilder(template=unified_template)
    prompt_result = prompt_builder.run(
        documents=retrieved_docs,
        query=query,
        topic=topic.replace('_', ' ').title(),
        domain=domain
    )
    
    # Step 4: Generate answer
    llm = OpenAIGenerator(
        api_key=Secret.from_env_var("OPENAI_API_KEY"),
        model="gpt-3.5-turbo",
        generation_kwargs={"temperature": 0.3}
    )
    answer_result = llm.run(prompt=prompt_result['prompt'])
    
    end_time = time.time()
    
    return {
        "question": query,
        "classified_topic": topic,
        "domain": domain,
        "answer": answer_result['replies'][0],
        "retrieved_docs": retrieved_docs,
        "response_time": end_time - start_time
    }

print("✅ Unified multi-topic RAG system ready!")

# Step 4: Test the unified system
print("\n" + "="*80)
print("TESTING MULTI-TOPIC RAG SYSTEM")
print("="*80)

test_questions = [
    # Company policy questions
    "How many vacation days do employees get?",
    "What is the parental leave policy?",
    
    # Python documentation questions
    "How do I handle errors in Python?",
    "What is a list comprehension?",
    
    # News questions
    "What are the recent developments in quantum computing?",
    "What's happening with electric vehicles?"
]

for question in test_questions:
    print(f"\n{'='*80}")
    print(f"❓ Question: {question}")
    print(f"{'='*80}")
    
    result = ask_unified_question(question, top_k=2)
    
    print(f"\n🏷️  Classified Topic: {result['classified_topic'].replace('_', ' ').title()}")
    print(f"📂 Domain: {result['domain']}")
    print(f"\n💡 Answer:\n{result['answer']}")
    print(f"\n⏱️  Response time: {result['response_time']:.2f}s")
    print(f"\n📄 Retrieved documents:")
    for i, doc in enumerate(result['retrieved_docs'], 1):
        title = doc.meta.get('title', f"Document {i}")
        print(f"   {i}. {title} (score: {doc.score:.3f})")

# Summary
print("\n\n" + "="*80)
print("MULTI-TOPIC RAG SYSTEM SUMMARY")
print("="*80)
print("""
✅ FEATURES IMPLEMENTED:

1. Unified Document Store:
   - Combined 3 different knowledge bases
   - Enhanced metadata with topic and domain tags
   - Single source of truth for all information

2. Intelligent Query Routing:
   - Automatic classification based on keywords
   - Routes to appropriate knowledge base
   - Fallback to default category for ambiguous queries

3. Filtered Retrieval:
   - Uses metadata filters to search specific topics
   - Improves relevance by focusing on correct domain
   - Reduces noise from irrelevant documents

4. Adaptive Responses:
   - Context-aware prompt includes topic and domain
   - Tone adapts based on subject matter
   - Clear source attribution

🎯 BENEFITS:

- Single system handles multiple domains
- Better accuracy through focused retrieval
- Scalable architecture for adding new topics
- Clear provenance and source tracking
- Improved user experience with automatic routing

🚀 NEXT STEPS:

- Add more sophisticated classification (ML-based)
- Implement hybrid retrieval (BM25 + embeddings)
- Add confidence scores for routing decisions
- Support multi-topic queries
- Add user feedback loop for improving classification
""")

PromptBuilder has 4 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


BUILDING MULTI-TOPIC RAG SYSTEM

📚 Step 1: Combining all knowledge bases...

✅ Combined 23 documents:
   - Company Policies: 13
   - Python Documentation: 6
   - News Articles: 4

📦 Unified document store created with 23 documents

🧭 Step 2: Implementing query routing logic...

✅ Query classification function created

🔧 Step 3: Building unified pipeline...

✅ Unified multi-topic RAG system ready!

TESTING MULTI-TOPIC RAG SYSTEM

❓ Question: How many vacation days do employees get?


PromptBuilder has 4 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.



🏷️  Classified Topic: Company Policy
📂 Domain: HR and Benefits

💡 Answer:
Full-time employees at TechCorp receive 20 days of paid vacation annually, accruing at a rate of 1.67 days per month. Unused vacation days can be carried over up to 5 days into the next year. (Source: Vacation Policy)

⏱️  Response time: 1.42s

📄 Retrieved documents:
   1. Vacation Policy (score: 10.191)
   2. Remote Work Policy (score: 5.809)

❓ Question: What is the parental leave policy?


PromptBuilder has 4 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


...


🏷️  Classified Topic: Technical Documentation
📂 Domain: Python Programming

💡 Answer:
In Python, you can handle errors using try-except blocks. This allows you to gracefully manage exceptions that may occur during the execution of your code. 

Here's a basic example of how to handle errors in Python:

```python
try:
    risky_operation()
except ExceptionType as e:
    handle_error(e)
```

In this code snippet, `risky_operation()` represents the code that may potentially raise an exception. If an exception of type `ExceptionType` occurs, it will be caught by the `except` block, and the `handle_error()` function will be called to manage the error.

Additionally, you can handle multiple exceptions by specifying them within the `except` block, like so:

```python
except (TypeError, ValueError) as e:
    handle_error(e)
```

Furthermore, you can use a `finally` block to execute code regardless of whether an exception occurs. This can be useful for cleanup operations.

If you encounter a si

PromptBuilder has 4 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.



🏷️  Classified Topic: Technical Documentation
📂 Domain: Python Programming

💡 Answer:
A list comprehension in Python is a concise way to create lists by applying operations to each element of another sequence. The syntax for a list comprehension is [expression for item in iterable if condition]. For example, you can create a list of squares of numbers from 0 to 9 using the expression squares = [x**2 for x in range(10)]. This information is sourced from the technical documentation on List Comprehensions.

⏱️  Response time: 1.42s

📄 Retrieved documents:
   1. List Comprehensions (score: 4.950)
   2. Dictionaries (score: 2.549)

❓ Question: What are the recent developments in quantum computing?


PromptBuilder has 4 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.



🏷️  Classified Topic: News
📂 Domain: Current Events

💡 Answer:
Recent developments in quantum computing include a major breakthrough announced by scientists at Tech University. They have achieved quantum supremacy with a 1000-qubit processor, allowing the new system to solve complex optimization problems 1000 times faster than classical supercomputers. This advancement has the potential to revolutionize fields such as drug discovery and climate modeling, as stated by lead researcher Dr. Smith. (Source: Major Breakthrough in Quantum Computing)

⏱️  Response time: 1.23s

📄 Retrieved documents:
   1. Major Breakthrough in Quantum Computing (score: 10.109)
   2. New Climate Agreement Reached (score: 6.567)

❓ Question: What's happening with electric vehicles?

🏷️  Classified Topic: News
📂 Domain: Current Events

💡 Answer:
Electric vehicle sales have seen a significant surge, with a 200% increase year-over-year. They now make up 25% of all new car sales globally. Major automakers have anno

## 11. Summary & Next Steps

### What You've Learned

✅ **RAG Fundamentals**
- What RAG is and when to use it
- Core components: Document Store, Retriever, Prompt Builder, Generator
- How components connect in a pipeline

✅ **Practical Implementation**
- Built 3 different RAG systems (HR, documentation, news)
- Worked with real-world use cases
- Handled different document types and structures

✅ **Prompt Engineering**
- Compared different prompt structures
- Learned best practices for RAG prompts
- Understood importance of clear instructions

✅ **Evaluation**
- Measured retrieval precision
- Analyzed response times
- Basic performance metrics

### Limitations of Basic RAG

The approach we've used has some limitations:

1. **Keyword-based retrieval (BM25)** misses semantic similarity
2. **No reranking** - first retrieved docs might not be most relevant
3. **Static pipeline** - no adaptive behavior
4. **No conversation memory** - each query is independent
5. **Limited evaluation** - we haven't measured answer quality systematically

### Next Steps

In **Notebook 2 - Advanced Retrieval Techniques**, you'll learn:

- 🔍 **Semantic search** with dense embeddings
- 🔄 **Hybrid retrieval** (BM25 + embeddings)
- 📈 **Reranking** for improved precision
- 🎯 **Query expansion** for better recall
- 📊 **Comprehensive evaluation** metrics

### Additional Resources

- [Haystack Documentation](https://docs.haystack.deepset.ai/)
- [RAG Paper (Lewis et al.)](https://arxiv.org/abs/2005.11401)
- [Prompt Engineering Guide](https://www.promptingguide.ai/)

---

**Congratulations!** You've completed the RAG Fundamentals notebook. You now have a solid foundation for building more sophisticated RAG systems.

Ready to level up? Continue to **Notebook 2: Advanced Retrieval Techniques**! 🚀

---

## Appendix: Helper Functions Reference

Here's a quick reference of all helper functions created in this notebook:

In [ ]:
# Helper functions summary
helper_functions = {
    "generate_company_knowledge_base()": "Creates synthetic company policy documents",
    "generate_python_docs()": "Creates Python documentation snippets",
    "generate_news_articles()": "Creates sample news articles",
    "ask_question(pipeline, question, top_k)": "Runs a query through RAG pipeline",
    "evaluate_retrieval_precision(pipeline, test_cases)": "Evaluates retrieval quality",
    "measure_response_times(pipeline, queries, num_runs)": "Measures pipeline latency",
    "compare_prompts(question, store, templates)": "Compares different prompt templates"
}

print("Helper Functions:")
for func, description in helper_functions.items():
    print(f"  • {func}: {description}")